# 03 · 函数与程序组织

把重复的逻辑包装成函数（function），并学会用参数、作用域与类型提示，让程序更清晰、可重用、好维护。

## 学习目标
- 能用 def 定义函数并以 return 回传结果，理解有无回传值的差异。
- 掌握位置参数（positional）、关键字参数（keyword）、默认值参数（default），以及 *args / **kwargs 的弹性传参方式。
- 理解作用域（scope）、嵌套函数（nested function）与闭包（closure），并能用 nonlocal 修改外层变量。
- 能写出纯函数（pure function）与递归（recursion），判断何时适用。
- 会用 lambda 匿名函数撰写简短函数，并用类型提示（type hints）标注参数与回传类型。
- 认识模块（module）的三种 import 方式，能合理组织代码。

## 函数基础

函数（function）是把一段可命名、可重复使用的逻辑封装起来的基本单位。
想避免同样的代码重复贴好几次、想让逻辑有个好懂的名字时就用它。

关键分辨：回传值（return）把结果交回给调用端，可以再被运算或存起来；印出（print）只是显示在画面上，函数本身回传的是 None。

形状：
```
def 函数名(参数):
    ...
    return 结果
```

In [ ]:
# 概念：def 定义函数、return 回传值，与「只 print 不 return」的差别
def c_to_f_return(celsius):
    return celsius * 9 / 5 + 32      # 把结果交回调用端，可再被使用

def c_to_f_print(celsius):
    print(celsius * 9 / 5 + 32)      # 只显示在画面上，没有交回结果

# 调用（call）有 return 的版本：结果能存进变量、继续算
f1 = c_to_f_return(25)
print('return 版拿到的值：', f1)
print('可继续运算，再加 1：', f1 + 1)

# 注意：只 print 的版本，回传的是 None，存起来没有可用数值
f2 = c_to_f_print(25)               # 这行会印出 77.0，但 f2 拿到的是 None
print('print 版回传的是：', f2)

## 参数的多种样貌

传参方式决定函数调用起来有多弹性。位置参数（positional argument）按顺序对应；关键字参数（keyword argument）用名字对应、顺序可变；默认值参数（default argument）让某些参数可省略。

当参数数量不固定时，用 *args 收集任意数量的位置参数成 tuple，用 **kwargs 收集任意数量的关键字参数成 dict。

顺序规则：一般参数 → 默认值参数 → *args → **kwargs。

In [ ]:
# 概念：位置参数、关键字参数、默认值参数，以及 *args / **kwargs
def greet(greeting, *names, punctuation='。', **options):
    # greeting 是必填位置参数；punctuation 有默认值可省略
    # *names 收集任意数量的称呼成 tuple；**options 收集额外设置成 dict
    who = '、'.join(names) if names else '大家'   # 没给名字就用「大家」
    line = f'{greeting}，{who}{punctuation}'
    for key, value in options.items():            # 把附加设置逐条附在后面
        line += f'（{key}={value}）'
    return line

# 只给必填参数：其余走预设
print(greet('早安'))

# 多个称呼进 *names，并用关键字参数覆写标点
print(greet('哈啰', '小明', '小华', punctuation='！'))

# 技巧：**kwargs 让调用端能丢入未事先枚举的设置
print(greet('您好', '王经理', mood='正式', lang='zh'))

## 纯函数与函数设计观念

纯函数（pure function）只依赖传入的参数、不改变外部状态，输入相同则输出永远相同。
相对地，会改到外部数据或环境的行为叫副作用（side effect）。

为什么追求纯函数：好测试、好重用、不会在别处偷偷改坏别人的数据，符合单一职责。学会辨识并避免不必要的副作用，是写出可维护程序的关键。

In [ ]:
# 概念：对照「有副作用」与「纯函数」两种清单加总写法
def add_value_impure(numbers, value):
    numbers.append(value)            # 注意：直接改到传入的清单，是副作用
    return sum(numbers)

def add_value_pure(numbers, value):
    new_numbers = numbers + [value]  # 造一份新清单，不动原始数据
    return sum(new_numbers)

# 造一组假数据当示范用
data = [10, 20, 30]

# 调用有副作用版本：原始 data 被改掉了
total_impure = add_value_impure(data, 40)
print('impure 加总：', total_impure, '；调用后原始 data：', data)

# 重置后调用纯函数版本：原始 data 维持不变
data = [10, 20, 30]
total_pure = add_value_pure(data, 40)
print('pure 加总：', total_pure, '；调用后原始 data：', data)

## 作用域、嵌套函数与闭包

作用域（scope）决定变量在哪里看得到。函数内定义的是区域变量（local），外面看不到；函数外的是全域变量（global）。Python 查找变量时由内往外一层层找。

嵌套函数（nested function）是函数里再定义函数。当内层函数记住并使用外层的变量，就形成闭包（closure）。预设情况下内层只能读外层变量；要在内层修改外层变量，需用 nonlocal 声明。

In [ ]:
# 概念：嵌套函数 + 闭包，用 nonlocal 在内层修改外层变量
def make_counter(start=0):
    count = start                    # 外层的区域变量，会被内层记住（形成闭包）

    def step():
        nonlocal count               # 注意：没有 nonlocal，内层赋值会被当成新的区域变量
        count += 1                    # 累加外层那个 count
        return count

    return step                      # 回传内层函数本身（连同它记住的 count）

# 每调用一次 make_counter 都会得到独立的计数器
counter = make_counter()
print('第一次：', counter())
print('第二次：', counter())
print('第三次：', counter())

# 技巧：另开一个计数器互不干扰，证明各自记住自己的 count
other = make_counter(start=100)
print('另一个计数器：', other())

## lambda 与递归

lambda 是匿名函数（anonymous function），适合写一行即用即丢的小函数，常拿来当排序的键（key）。

形状：
```
lambda 参数: 表达式
```

递归（recursion）是函数调用自己，把问题拆成更小的同类问题。一定要设好终止条件（base case），否则会无限递归而崩溃。

In [ ]:
# 概念：lambda 当排序键、递归计算阶乘与终止条件
words = ['banana', 'fig', 'cherry', 'kiwi']

# 用 lambda 取每个字符串的长度当排序依据（即用即丢，不必另外 def）
by_length = sorted(words, key=lambda w: len(w))
print('依长度排序：', by_length)

def factorial(n):
    if n <= 1:                       # 终止条件：到 1 就停，避免无限递归
        return 1
    return n * factorial(n - 1)      # 把问题缩小一阶，逐步收敛到终止条件

print('5 的阶乘：', factorial(5))
print('逐步展开 3! =', '3 * 2 * 1 =', factorial(3))

## 类型提示

类型提示（type hints）在参数与回传值上标注预期类型。它不影响程序运行，但能让函数接口更清楚、利于编辑器与检查工具找错、也方便阅读。

容器与特殊情况常用 typing 模块：List、Dict 标注容器内容，Optional[X] 表示可能是 X 也可能是 None（例如查不到数据），Any 表示任意类型。

形状：
```
def 函数名(参数: 类型) -> 回传类型:
```

In [ ]:
# 概念：用 type hints 标注参数与回传类型，Optional 表示可能找不到
from typing import Optional, Dict, List

# 造一份假的用户数据库当示范用
users: List[Dict[str, str]] = [
    {'id': 'u01', 'name': '小明'},
    {'id': 'u02', 'name': '小华'},
]

def find_user(db: List[Dict[str, str]], user_id: str) -> Optional[Dict[str, str]]:
    # 回传类型用 Optional：找到回传 dict，找不到回传 None
    for record in db:
        if record['id'] == user_id:
            return record
    return None

print('查到的用户：', find_user(users, 'u02'))
print('查不到时回传：', find_user(users, 'u99'))

# 技巧：类型提示只是注记，可用 __annotations__ 查看，不会强制运行
print('函数的类型注记：', find_user.__annotations__)

## 模块与 import

模块（module）是一个 .py 档，里面装着可重用的函数与变量。把程序拆成模块，能避免单档过长、方便重用。

三种 import 写法各有取舍：

| 写法 | 调用方式 | 适用情境 |
|---|---|---|
| `import math` | `math.sqrt(x)` | 最清楚，明示来源，避免名称冲突 |
| `import numpy as np` | `np.array(...)` | 模块名太长时取别名 |
| `from math import sqrt` | `sqrt(x)` | 只用少数几个名字、想写短一点 |

命名空间（namespace）就是「名字所属的容器」，用 `math.` 这种前缀可避免不同模块的同名函数互相盖掉。

In [ ]:
# 概念：三种 import 写法的差异与调用名称长短
import math                          # 写法一：保留命名空间，math.xxx
import random as rnd                 # 写法二：取别名，rnd.xxx
from math import pi                  # 写法三：直接取名字进来，pi

print('math.sqrt(16) =', math.sqrt(16))      # 来源清楚
print('rnd 取别名后 seed 固定再取数：')
rnd.seed(0)                          # 固定乱数种子，让示范输出可重现
print('  random 值 =', round(rnd.random(), 4))
print('直接用 pi =', pi)

# 注意：from X import * 会把所有名字灌进来，易造成名称冲突，正式程序少用
print('math.floor(pi) =', math.floor(pi))

## 练习

以下三题由浅入深，皆为复合型但简短。数据请在题目内自己用 list 造（仿真即可），不要引用外部文件。每题完成后对照「验收」确认结果。

In [ ]:
# TODO 1 ·（简单）楼地板面积计算函数（集成：函数基础 + 类型提示）
#   情境：给定几栋建筑的占地面积（footprint area）与楼层数（floors），
#         要算出各栋的楼地板面积 GFA（gross floor area）。
#   要求：
#     1. 用 list 自己造 4 栋建筑的数据：占地面积（平方公尺）与楼层数两个清单（仿真数字即可）。
#     2. 定义函数 gfa(footprint: float, floors: int) -> float，加上类型提示，
#        以 return 回传「占地面积 × 楼层数」。
#     3. 用循环调用该函数，把每栋的 GFA 收进一个 list。
#   验收：应该看到 4 个 GFA 数值，且每个都等于对应的占地面积乘以楼层数。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）可设置的建筑筛选统计（集成：参数的多种样貌 + 纯函数 + 类型提示 + 模块 import）
#   情境：一批建筑数据含楼高（height）、楼层数（floors）与用途分类标签（use），
#         要做可调条件的筛选与汇总。
#   要求：
#     1. 用 list 造一组建筑数据（每栋是一个 dict，含 height、floors、use 三个键），数字仿真即可。
#     2. 定义纯函数 summarize(buildings, min_height: float = 0.0, **filters)：
#        用默认值参数设高度门槛，用 **kwargs 接收额外等值筛选（例如 use='住宅'）；
#        函数不可修改传入的原始数据。
#     3. 函数回传一个 Dict（加类型提示），内含符合条件的栋数、平均楼高、最高楼层数。
#     4. 在主程序 import math（任一种 import 写法），用它做四舍五入或开根号等其中一个收尾计算。
#   验收：改变 min_height 或 **filters 后，回传的栋数与平均值跟着变，且原始数据清单内容不变。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）街廓网格的容积聚合与政策情境比较（集成：闭包 + nonlocal + 递归 / 聚合 + lambda + 纯函数）
#   情境：把城市切成数个街廓网格 cell，每格内有若干建筑，要聚合各格的总楼地板面积，
#         并比较「放宽容积率 FAR（floor area ratio）」政策前后的变化。
#   要求：
#     1. 用嵌套 list 造数据：数个 cell，每个 cell 是一串建筑的 (footprint, floors)（仿真数字即可）。
#     2. 写一个外层函数回传闭包累加器，内层用 nonlocal 累加跑过的总 GFA；
#        用它走访所有 cell 聚合出全市总量（可用递归或循环走访嵌套结构）。
#     3. 用 lambda 定义政策乘数（例如 apply = lambda gfa, k: gfa * k），
#        以纯函数方式对每格 GFA 套用放宽倍率 k，产生情境后的新值（不改原始数据）。
#     4. 比较政策前后的全市总 GFA，并找出增量最大的那一格。
#   验收：政策后总量大于政策前，且能正确指出增幅最大的 cell 编号。
# TODO: 在这里完成

## 小结

- 函数用 def 定义、用 return 把结果交回调用端；return 与 print 是两回事，没有 return 时回传 None。
- 参数有位置、关键字、默认值三种传法，*args / **kwargs 收集不定量参数，顺序规则固定。
- 纯函数只依赖参数、不改外部状态，较好测试与重用；要警觉并避免不必要的副作用。
- 内层函数可记住外层变量形成闭包，nonlocal 让内层能修改外层变量。
- lambda 写一行即用即丢的小函数；递归一定要设终止条件才会收敛。
- 类型提示让接口更清楚但不影响运行；用 Optional / List / Dict 标注特殊与容器类型。
- 三种 import 写法各有取舍，命名空间能避免名称冲突，模块让程序好拆好重用。